# ⛽ Gas Price Correlation Analysis

This notebook explores correlations between U.S. gas prices and related economic factors including:
- Crude oil prices (WTI)
- Inflation (CPI)
- Unemployment rate
- Seasonal patterns

**Data source:** Simulated data based on realistic historical trends (swap in real CSVs from EIA, FRED, or Kaggle).

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded ✅')

## 2. Load Data

> 💡 **Swap this section** with real data from:
> - [EIA Gas Prices](https://www.eia.gov/petroleum/gasdiesel/)
> - [FRED Economic Data](https://fred.stlouisfed.org/)
> - [Kaggle Gas Price Datasets](https://www.kaggle.com/search?q=gas+prices)

In [ ]:
np.random.seed(42)
n = 240

dates = pd.date_range(start='2004-01-01', periods=n, freq='MS')

crude = 50 + np.cumsum(np.random.randn(n) * 2) + 20 * np.sin(np.linspace(0, 6 * np.pi, n))
crude = np.clip(crude, 20, 120)

gas_price = 1.5 + crude * 0.025 + np.random.randn(n) * 0.15
gas_price = np.clip(gas_price, 1.5, 5.5)

cpi = 180 + np.linspace(0, 120, n) + np.random.randn(n) * 3

unemployment = 5 + 2 * np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.randn(n) * 0.5
unemployment = np.clip(unemployment, 3, 14)

df = pd.DataFrame({
    'date': dates,
    'gas_price_usd': gas_price,
    'crude_oil_wti': crude,
    'cpi': cpi,
    'unemployment_rate': unemployment
}).set_index('date')

df['month'] = df.index.month
df['year'] = df.index.year

df.head(10)

## 3. Overview & Summary Stats

In [ ]:
print(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cols = {
    'gas_price_usd': 'Gas Price ($/gal)',
    'crude_oil_wti': 'Crude Oil WTI ($/bbl)',
    'cpi': 'CPI',
    'unemployment_rate': 'Unemployment Rate (%)'
}
for ax, (col, label) in zip(axes.flat, cols.items()):
    ax.plot(df.index, df[col], linewidth=1.5)
    ax.set_title(label)

plt.suptitle('Economic Indicators Over Time (2004–2024)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
corr_cols = ['gas_price_usd', 'crude_oil_wti', 'cpi', 'unemployment_rate']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Correlation Matrix: Gas Price vs Economic Factors')
plt.tight_layout()
plt.show()

## 5. Gas Price vs Crude Oil (Scatter + Regression)

In [ ]:
slope, intercept, r, p, se = stats.linregress(df['crude_oil_wti'], df['gas_price_usd'])

plt.figure(figsize=(9, 5))
sns.regplot(
    data=df, x='crude_oil_wti', y='gas_price_usd',
    scatter_kws={'alpha': 0.4, 's': 20},
    line_kws={'color': 'crimson', 'linewidth': 2}
)
plt.title(f'Gas Price vs Crude Oil  |  r = {r:.2f},  p < 0.001')
plt.xlabel('Crude Oil WTI ($/bbl)')
plt.ylabel('Gas Price ($/gal)')
plt.tight_layout()
plt.show()

print(f'Slope:     ${slope:.4f} per barrel')
print(f'Intercept: ${intercept:.4f}')
print(f'R²:        {r**2:.4f}')

## 6. Seasonal Patterns

In [ ]:
monthly_avg = df.groupby('month')['gas_price_usd'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(10, 4))
plt.bar(month_labels, monthly_avg, color=sns.color_palette('muted', 12))
plt.title('Average Gas Price by Month (Seasonal Pattern)')
plt.ylabel('Avg Gas Price ($/gal)')
plt.ylim(monthly_avg.min() * 0.95, monthly_avg.max() * 1.05)
plt.tight_layout()
plt.show()

## 7. Rolling Correlation (Gas Price vs CPI)

In [ ]:
rolling_corr = df['gas_price_usd'].rolling(24).corr(df['cpi'])

plt.figure(figsize=(12, 4))
plt.plot(df.index, rolling_corr, color='steelblue', linewidth=1.5)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('24-Month Rolling Correlation: Gas Price vs CPI')
plt.ylabel('Pearson r')
plt.tight_layout()
plt.show()

## 8. Key Findings

| Factor | Correlation with Gas Price | Notes |
|---|---|---|
| Crude Oil (WTI) | Strong positive | Primary driver |
| CPI | Moderate positive | Inflation effect |
| Unemployment | Weak / negative | Counter-cyclical |
| Month (season) | Weak | Summer peaks |

### Next Steps
- Replace simulated data with real EIA/FRED datasets
- Add regional breakdowns (state-level gas prices)
- Build a simple predictive model (linear regression or ARIMA)
- Analyze lag effects (crude → gas price delay)